# 09 — Ensemble Methods
**Referencias:** ESL Cap. 10 (Boosting), Cap. 15 (Random Forests) · Géron Cap. 7

## Teoría de Ensembles (ESL 8.8)
La sabiduría de las masas: combinar modelos débiles reduce la varianza sin aumentar bias.

**Bagging** (Bootstrap AGGregatING): reduce varianza promediando modelos entrenados en subsets bootstrap.
$$\hat{f}_{\text{bag}}(x) = \frac{1}{B}\sum_{b=1}^B \hat{f}^b(x)$$

**Boosting** (ESL Cap. 10): reduce bias ajustando residuales secuencialmente.
$$F_m(x) = F_{m-1}(x) + \gamma_m h_m(x)$$

In [ ]:
import os
os.chdir('/Volumes/SSD_Gabo/proyectos/growth-analytics')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import (
    BaggingClassifier, RandomForestClassifier, ExtraTreesClassifier,
    GradientBoostingClassifier, AdaBoostClassifier,
    VotingClassifier, StackingClassifier
)
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, cross_val_score, learning_curve
from sklearn.metrics import roc_auc_score
from sklearn.inspection import permutation_importance

plt.rcParams.update({
    'figure.dpi': 130, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.color': '#e8e8e8', 'axes.axisbelow': True,
    'axes.titlesize': 11, 'axes.titleweight': 'bold', 'legend.frameon': False,
})
np.random.seed(42)
COLORS = ['#4361ee','#f72585','#06d6a0','#ff9f1c','#7209b7']

n = 3000
sessions      = np.random.randint(1, 50, n).astype(float)
days_active   = np.random.randint(1, 90, n).astype(float)
transactions  = np.random.randint(0, 30, n).astype(float)
time_on_site  = np.random.uniform(30, 600, n)
support_calls = np.random.poisson(1, n).astype(float)

logit = -3 + sessions*0.05 + days_active*0.02 + transactions*0.08 + time_on_site*0.002
y = (np.random.uniform(0,1,n) < 1/(1+np.exp(-logit))).astype(int)

X = np.column_stack([sessions, days_active, transactions, time_on_site, support_calls])
feat_names = ['sessions','days_active','transactions','time_on_site','support_calls']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f'Train: {X_train.shape}, Activation rate: {y_train.mean():.2%}')

## 1 — Bagging y el Out-of-Bag score (ESL Cap. 8 / Géron Cap. 7)

In [ ]:
# OOB: en cada árbol, ~37% de muestras no se usan (out-of-bag)
# Se pueden usar como validation set gratis

rf_oob = RandomForestClassifier(
    n_estimators=200, oob_score=True, random_state=42, n_jobs=-1
)
rf_oob.fit(X_train, y_train)

oob_acc = rf_oob.oob_score_
test_auc = roc_auc_score(y_test, rf_oob.predict_proba(X_test)[:,1])

print('OOB Score (ESL 8.7) — estimación gratuita del error de generalización:')
print(f'  OOB accuracy: {oob_acc:.4f}')
print(f'  Test AUC:     {test_auc:.4f}')
print()

# Convergencia con número de árboles
n_trees = range(10, 201, 10)
oob_errs = []
for n_t in n_trees:
    rf_t = RandomForestClassifier(n_estimators=n_t, oob_score=True, random_state=42, n_jobs=-1)
    rf_t.fit(X_train, y_train)
    oob_errs.append(1 - rf_t.oob_score_)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(list(n_trees), oob_errs, 'o-', color='#4361ee', linewidth=2)
ax.set_xlabel('N árboles'); ax.set_ylabel('OOB Error')
ax.set_title('Convergencia del RF — OOB Error vs N Árboles')
# Marcar dónde estabiliza
stable_idx = np.argmin(np.diff(oob_errs[::-1])[::-1] > 0.001) + 1
ax.axvline(list(n_trees)[stable_idx], color='#f72585', linestyle='--',
           label=f'Estabiliza ~{list(n_trees)[stable_idx]} árboles')
ax.legend()
plt.tight_layout()
plt.show()

## 2 — Gradient Boosting: teoría y early stopping (ESL Cap. 10)

In [ ]:
# ESL Cap. 10.8: el learning rate y n_estimators se tradeoff
# Más árboles + learning_rate pequeño → mejor generalización

gb = GradientBoostingClassifier(
    n_estimators=500, learning_rate=0.05,
    max_depth=3, subsample=0.8,  # subsample < 1 → Stochastic GBM
    random_state=42,
    validation_fraction=0.15, n_iter_no_change=20, tol=1e-4  # early stopping
)
gb.fit(X_train, y_train)

print(f'Early stopping: paró en {gb.n_estimators_} de 500 árboles posibles')

# Curva de train vs validation loss
train_scores = gb.train_score_
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(train_scores, color='#4361ee', linewidth=1.5, label='Train loss')
ax.axvline(gb.n_estimators_ - 1, color='#f72585', linestyle='--',
           label=f'Early stop ({gb.n_estimators_} árboles)')
ax.set_xlabel('Iteración'); ax.set_ylabel('Log Loss')
ax.set_title('Gradient Boosting — Early Stopping (ESL 10.12)')
ax.legend()
plt.tight_layout()
plt.show()

gb_auc = roc_auc_score(y_test, gb.predict_proba(X_test)[:,1])
print(f'GBM AUC: {gb_auc:.4f}')

## 3 — Comparación de ensembles

In [ ]:
models = [
    ('Decision Tree (base)',     DecisionTreeClassifier(max_depth=3, random_state=42)),
    ('Bagging (100 trees)',      BaggingClassifier(n_estimators=100, random_state=42, n_jobs=-1)),
    ('Random Forest',            RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)),
    ('Extra Trees',              ExtraTreesClassifier(n_estimators=100, random_state=42, n_jobs=-1)),
    ('AdaBoost',                 AdaBoostClassifier(n_estimators=100, random_state=42)),
    ('Gradient Boosting',        GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, random_state=42)),
]

results = []
for name, model in models:
    scores = cross_val_score(model, X_train, y_train, cv=5, scoring='roc_auc', n_jobs=-1)
    results.append({'model': name, 'auc_mean': scores.mean(), 'auc_std': scores.std()})

results_df = pd.DataFrame(results).sort_values('auc_mean')

fig, ax = plt.subplots(figsize=(10, 5))
colors_bar = [COLORS[0] if 'Forest' in r or 'Boosting' in r or 'Trees' in r else '#adb5bd'
              for r in results_df['model']]
bars = ax.barh(results_df['model'], results_df['auc_mean'], xerr=results_df['auc_std'],
               color=colors_bar, alpha=0.85, capsize=5)
for bar, val in zip(bars, results_df['auc_mean']):
    ax.text(val + 0.002, bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center', fontsize=9)
ax.set_xlabel('AUC (5-Fold CV)')
ax.set_title('Ensemble Methods — Comparación (ESL Cap. 10, 15)')
ax.set_xlim(0.5, results_df['auc_mean'].max() + 0.04)
plt.tight_layout()
plt.show()

## 4 — Voting y Stacking (Géron Cap. 7)

In [ ]:
# Voting: combina predicciones con voto mayoritario o promedio de probabilidades
rf_clf   = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
gb_clf   = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, random_state=42)
lr_pipe  = Pipeline([('sc', StandardScaler()), ('lr', LogisticRegression(max_iter=1000))])

voting_soft = VotingClassifier(
    estimators=[('rf', rf_clf), ('gb', gb_clf), ('lr', lr_pipe)],
    voting='soft'
)
voting_hard = VotingClassifier(
    estimators=[('rf', rf_clf), ('gb', gb_clf), ('lr', lr_pipe)],
    voting='hard'
)

# Stacking: usa un meta-modelo para combinar predicciones de base models
stacking = StackingClassifier(
    estimators=[('rf', rf_clf), ('gb', gb_clf), ('lr', lr_pipe)],
    final_estimator=LogisticRegression(max_iter=1000),
    cv=5, passthrough=False
)

ensemble_models = [
    ('Random Forest',   rf_clf),
    ('Gradient Boost',  gb_clf),
    ('Logistic Reg',    lr_pipe),
    ('Voting (soft)',   voting_soft),
    ('Stacking',        stacking),
]

ensemble_results = []
for name, model in ensemble_models:
    model.fit(X_train, y_train)
    auc = roc_auc_score(y_test, model.predict_proba(X_test)[:,1])
    ensemble_results.append({'model': name, 'auc': auc})

ens_df = pd.DataFrame(ensemble_results).sort_values('auc')
fig, ax = plt.subplots(figsize=(9, 4))
colors_b = ['#4361ee' if 'Voting' in r or 'Stack' in r else '#adb5bd' for r in ens_df['model']]
bars = ax.barh(ens_df['model'], ens_df['auc'], color=colors_b, alpha=0.85)
for bar, val in zip(bars, ens_df['auc']):
    ax.text(val+0.001, bar.get_y()+bar.get_height()/2, f'{val:.4f}', va='center', fontsize=9)
ax.set_xlabel('AUC (test)')
ax.set_title('Voting y Stacking — Géron Cap. 7')
ax.set_xlim(0.5, ens_df['auc'].max() + 0.04)
plt.tight_layout()
plt.show()

## 5 — Feature Importance: Impurity vs Permutation (ESL Cap. 10 / Géron Cap. 7)

In [ ]:
rf_final = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
rf_final.fit(X_train, y_train)

# Impurity-based (rápida pero sesgada a features de alta cardinalidad)
imp_impurity = pd.Series(rf_final.feature_importances_, index=feat_names).sort_values()

# Permutation importance (no sesgada, más lenta)
perm = permutation_importance(rf_final, X_test, y_test, n_repeats=15, random_state=42, n_jobs=-1)
imp_permutation = pd.Series(perm.importances_mean, index=feat_names).sort_values()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].barh(imp_impurity.index, imp_impurity.values, color='#adb5bd')
axes[0].set_title('Impurity-based Importance (RF)\nsesgada a features con muchas divisiones')

colors_perm = ['#4361ee' if v > 0 else '#f72585' for v in imp_permutation.values]
axes[1].barh(imp_permutation.index, imp_permutation.values, color=colors_perm,
             xerr=pd.Series(perm.importances_std, index=feat_names).loc[imp_permutation.index])
axes[1].axvline(0, color='#888', linewidth=0.8)
axes[1].set_title('Permutation Importance (en test)\nno sesgada — ESL Cap. 10 / Géron')

plt.tight_layout()
plt.show()

## Resumen

| Método | Reduce | Mecanismo | Referencia |
|---|---|---|---|
| Bagging | Varianza | Promediar modelos en subsets bootstrap | ESL 8.7 |
| Random Forest | Varianza | Bagging + aleatoriedad en features | ESL Cap. 15 |
| AdaBoost | Bias | Reponderar errores secuencialmente | ESL 10.1 |
| Gradient Boosting | Bias | Ajustar residuales con árboles | ESL Cap. 10 |
| Voting (soft) | Varianza + bias | Promediar probabilidades | Géron Cap. 7 |
| Stacking | Varianza + bias | Meta-modelo aprende a combinar | Géron Cap. 7 |

**Siguiente:** `10_real_world.ipynb`